# Bicing Barcelona — Clustering de estaciones por comportamiento operativo

**Datos:** Open Data Barcelona · Tabla `features_clustering` (548 estaciones × 32 variables)  2022-2024

---

Este notebook agrupa las 548 estaciones de Bicing en clusters según su comportamiento operativo (disponibilidad, variabilidad, saturación) usando **K-Means**:

1. **Carga e inspección** — conexión a DuckDB y análisis de valores faltantes
2. **Preparación** — Filtrado de estaciones y eliminación de estaciones con demasiados NaN y Normalización
3. **Selección de k** — Método del codo (Elbow) + Silhouette
4. **Clustering K-Means con k=5** — Entrenamiento y asignación de clusters e interpretación de perfiles
5. **Visualización** — perfiles NAB, heatmap de centroides, distribución por distrito y mapa interactivo
6. **Exportación** — CSV y tabla DuckDB con los resultados

Cada sección genera gráficos (PNG) y datos (CSV) y se guardan en subcarpetas.

---
## 0. Configuración

Importamos las bibliotecas, conectamos a la base de datos DuckDB y definimos los parámetros globales que se utilizan a lo largo de todo el notebook.


In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import seaborn as sns
import folium
from branca.element import Element
import os
import glob
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')

# ── Estilo global de Matplotlib ─────────────────────────────────────────────
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'figure.dpi':       150,
    'savefig.dpi':      150,
    'savefig.bbox':     'tight',
    'axes.spines.top':  False,
    'axes.spines.right': False,
})

# ── Colores para los clusters ────────────────────────────────────────────────
COLORS_CLUSTER = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']

# ── Carpetas de salida ───────────────────────────────────────────────────────
for carpeta in ['png', 'csv', 'html']:
    os.makedirs(carpeta, exist_ok=True)

# ── Conexión a DuckDB ────────────────────────────────────────────────────────
DB_PATH = 'bicicletes.duckdb'
con = duckdb.connect(DB_PATH)
print('Connexió establerta a', DB_PATH)

---
## 1. Carga e inspección de los datos

Cargamos la tabla `features_clustering`, que contiene 32 columnas con métricas operativas para cada una de las 548 estaciones (2022–2024).


In [ ]:
# ── Carga de la tabla de variables ───────────────────────────────────────────
df = con.execute("SELECT * FROM features_clustering").df()

print(f"Dimensions: {df.shape[0]} estacions × {df.shape[1]} columnes")
display(df.head(3))

---
## 2. Análisis de valores faltantes (NaN)

Antes de hacer el clustering, hay que entender por qué algunas estaciones tienen datos incompletos y no pueden participar en el análisis.


In [ ]:
# ── Identificamos las columnas numéricas (variables) ─────────────────────────
# Excluimos id_estacio, lat, lon que son identificadores/coordenadas
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['id_estacio', 'lat', 'lon']]

print(f"Columnes numèriques (features): {len(numeric_cols)}")
print()

# ── Distribución de NaN por estación ────────────────────────────────────────
nan_per_row = df[numeric_cols].isnull().sum(axis=1)

print("Distribució de NaN per estació:")
print("(quantes estacions tenen 0, 1, 2... valors NaN)")
print()
dist = nan_per_row.value_counts().sort_index()
for n_nan, count in dist.items():
    print(f"  {int(n_nan):2d} NaN → {count} estacions")
print(f"\nTotal estacions: {len(df)}")

In [ ]:
# ── Columnas con valores NaN ─────────────────────────────────────────────────
nan_per_col = df[numeric_cols].isnull().sum()
cols_with_nan = nan_per_col[nan_per_col > 0].sort_values(ascending=False)

if len(cols_with_nan) > 0:
    print("Columnes amb valors NaN:")
    print()
    for col, count in cols_with_nan.items():
        print(f"  {col}: {count} estacions sense dada")
else:
    print("Cap columna amb NaN")

In [ ]:
# ── Criterio de eliminación ──────────────────────────────────────────────────
# KMeans necesita un valor numérico en CADA variable para calcular distancias.
# Si una estación tiene la mayoría de variables vacías, no tenemos información
# suficiente para asignarla a ningún cluster.
#
# Criterio: eliminamos estaciones con más de 10 NaN (de 28 variables).
# Esto corresponde a estaciones nuevas, sin suficiente historial 2022-2024.

estacions_invalides = df[nan_per_row > 10][['id_estacio', 'nom_districte']].copy()
estacions_invalides['n_nan'] = nan_per_row[nan_per_row > 10].values

print(f"Estacions eliminades (>10 NaN de 28 features): {len(estacions_invalides)}")
print(f"Estacions vàlides per al clustering: {len(df) - len(estacions_invalides)}")
print()
print("Detall de les estacions eliminades:")
display(estacions_invalides.sort_values('n_nan', ascending=False))

---
## 3. Filtrado y preparación de los datos

Eliminamos las estaciones con demasiados NaN e imputamos los pocos NaN restantes con la mediana.


In [ ]:
# ── Filtrar estaciones inválidas ─────────────────────────────────────────────
df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
df['lon'] = pd.to_numeric(df['lon'], errors='coerce')
df_valid = df[nan_per_row <= 10].copy()
df_valid['lat'] = pd.to_numeric(df_valid['lat'], errors='coerce')
df_valid['lon'] = pd.to_numeric(df_valid['lon'], errors='coerce')

# ── Imputar NaN restantes con la mediana ─────────────────────────────────────
for col in numeric_cols:
    if df_valid[col].isnull().any():
        df_valid[col] = df_valid[col].fillna(df_valid[col].median())

print(f"Estacions per al clustering: {len(df_valid)}")
print(f"NaN restants: {df_valid[numeric_cols].isnull().sum().sum()}")

---
## 4. Normalización de las variables

Aplicamos `StandardScaler` (z-score) para que todas las variables tengan el mismo peso en el cálculo de distancias. Sin normalizar, las variables con valores grandes dominarían el clustering.


In [ ]:
# ── Normalización z-score ────────────────────────────────────────────────────
feature_cols = numeric_cols  # 28 features

X = df_valid[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Features normalitzades: {len(feature_cols)}")
for i, f in enumerate(feature_cols):
    print(f"  {i+1:2d}. {f}")

---
## 5. Selección del número de clusters

Probamos k de 2 a 12 y observamos dos métricas:
- **Inercia (Elbow):** suma de distancias dentro de los clusters. Buscamos el "codo" donde la mejora se reduce.
- **Silhouette:** mide qué tan bien separados están los clusters (máximo = 1, mínimo = −1).


In [ ]:
# ── Métricas para k = 2..12 ──────────────────────────────────────────────────
k_range = range(2, 13)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

metrics_df = pd.DataFrame({
    'k': list(k_range),
    'inertia': inertias,
    'silhouette': silhouettes,
})
print(metrics_df.to_string(index=False))

In [ ]:
# ── Gráfico Elbow + Silhouette ───────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#F8F9FA'); ax1.set_facecolor('#F8F9FA')
k_list = list(k_range)

ax1.plot(k_list, inertias, 'b-o', linewidth=2.5, label='Inèrcia (SSE)')
ax1.set_xlabel('Nombre de clusters (k)', fontsize=10)
ax1.set_ylabel('Inèrcia', color='b', fontsize=10)
ax1.tick_params(axis='y', labelcolor='b')
ax1.axvline(x=5, color='gray', linestyle='--', alpha=0.6, label='k=5 escollit')

ax2 = ax1.twinx()
ax2.plot(k_list, silhouettes, 'r--s', linewidth=2.5, label='Silhouette')
ax2.set_ylabel('Silhouette Score', color='r', fontsize=10)
ax2.tick_params(axis='y', labelcolor='r')

# Leyenda combinada
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9, framealpha=.9)

ax1.set_title('Selecció de k: Elbow + Silhouette Score',
              fontsize=13, fontweight='bold', pad=14, color='#1D3557')
ax1.grid(axis='y', ls='--', alpha=.3)

plt.tight_layout()
plt.savefig('png/elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardat: png/elbow_silhouette.png')

metrics_df.to_csv('csv/cluster_metrics.csv', index=False)
print('Guardat: csv/cluster_metrics.csv')

---
## 6. Clustering KMeans con k=5

Elegimos k=5 porque ofrece un buen equilibrio entre inercia, silhouette e interpretabilidad de los resultados.


In [ ]:
# ── KMeans final con k=5 ─────────────────────────────────────────────────────
K_FINAL = 5

km = KMeans(n_clusters=K_FINAL, random_state=42, n_init=20, max_iter=500)
labels = km.fit_predict(X_scaled)
df_valid['cluster'] = labels

print("Estacions per cluster:")
print(df_valid['cluster'].value_counts().sort_index())
print(f"\nSilhouette (k={K_FINAL}): {silhouette_score(X_scaled, labels):.4f}")

---
## 7. Interpretación y nombres de los clusters

Analizamos los centroides (valores medios de cada cluster en escala original) para asignar nombres descriptivos.


In [ ]:
# ── Centroides en escala original ────────────────────────────────────────────
centroids_scaled = km.cluster_centers_
centroids_orig = scaler.inverse_transform(centroids_scaled)
centroids_df = pd.DataFrame(centroids_orig, columns=feature_cols)
centroids_df['cluster'] = range(K_FINAL)

# Variables clave para interpretar
key_cols = ['capacitat_mitja', 'mitja_nab_global', 'std_nab_global',
            'ratio_buida_global', 'ratio_plena_global',
            'ratio_buida_hora_punta', 'nab_mitja_punta_matinada', 'nab_mitja_punta_tarda']

print("Centroides — features clau:")
display(centroids_df[['cluster'] + key_cols].round(3))

In [ ]:
# ── Nombres descriptivos basados en el comportamiento observado ───────────────
cluster_names = {
    0: "Perifèric Baix Ús",
    1: "Central Alta Ocupació",
    2: "Origen Commuter Matí",
    3: "Ús Mixt Equilibrat",
    4: "Residencial Entrada Tarda"
}

df_valid['cluster_name'] = df_valid['cluster'].map(cluster_names)
centroids_df['cluster_name'] = centroids_df['cluster'].map(cluster_names)
centroids_df['n_stations'] = [
    (df_valid['cluster'] == c).sum() for c in range(K_FINAL)
]

print("Resum per cluster:")
print(df_valid.groupby('cluster_name').size().reset_index(name='n_estacions'))

---
## 8. Perfiles NAB por cluster

Visualización del nivel medio de bicicletas disponibles (NAB) a lo largo del día, para días laborables y festivos. Esto nos ayuda a entender cómo cada cluster consume y libera bicicletas.


In [ ]:
# ── Perfiles NAB por cluster: laborables vs. festivos ────────────────────────
lab_slots = ['06-08', '08-10', '10-12', '12-14', '14-16', '16-18', '18-20', '20-22', '22-00']
lab_cols = ['nab_lab_h06_08', 'nab_lab_h08_10', 'nab_lab_h10_12', 'nab_lab_h12_14',
            'nab_lab_h14_16', 'nab_lab_h16_18', 'nab_lab_h18_20', 'nab_lab_h20_22', 'nab_lab_h22_00']

fest_slots = ['06-10', '10-14', '14-18', '18-22', '22-00']
fest_cols = ['nab_fest_h06_10', 'nab_fest_h10_14', 'nab_fest_h14_18', 'nab_fest_h18_22', 'nab_fest_h22_00']

# Estilos de línea diferenciados por cluster
dash_styles = ['-', '--', '-.', ':', (0, (5, 1))]
markers = ['o', 's', '^', 'D', 'v']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#F8F9FA')
for ax in axes:
    ax.set_facecolor('#F8F9FA')

for c in range(K_FINAL):
    lab_vals = [centroids_df.loc[c, col] for col in lab_cols]
    fest_vals = [centroids_df.loc[c, col] for col in fest_cols]

    axes[0].plot(lab_slots, lab_vals, linestyle=dash_styles[c], marker=markers[c],
                 color=COLORS_CLUSTER[c], linewidth=2.5, label=f"C{c}: {cluster_names[c]}")
    axes[1].plot(fest_slots, fest_vals, linestyle=dash_styles[c], marker=markers[c],
                 color=COLORS_CLUSTER[c], linewidth=2.5, label=f"C{c}: {cluster_names[c]}")

axes[0].set_title('Perfil NAB — Dies laborables', fontsize=11, fontweight='bold', pad=10, color='#1D3557')
axes[0].set_xlabel('Franja horària', fontsize=9)
axes[0].set_ylabel('NAB mig (0=buit, 1=ple)', fontsize=9)
axes[0].set_ylim(0, 0.85)
axes[0].legend(fontsize=8, framealpha=.9)
axes[0].tick_params(axis='x', rotation=30)
axes[0].grid(axis='y', ls='--', alpha=.3)

axes[1].set_title('Perfil NAB — Dies festius', fontsize=11, fontweight='bold', pad=10, color='#1D3557')
axes[1].set_xlabel('Franja horària', fontsize=9)
axes[1].set_ylabel('NAB mig (0=buit, 1=ple)', fontsize=9)
axes[1].set_ylim(0, 0.85)
axes[1].legend(fontsize=8, framealpha=.9)
axes[1].grid(axis='y', ls='--', alpha=.3)

fig.text(.5, -.03, 'Font: Open Data Barcelona · Servei Bicicletes · Elaboració pròpia',
         ha='center', fontsize=6, color='#666')

plt.tight_layout()
plt.savefig('png/nab_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardat: png/nab_profiles.png')

### 9. Heatmap de centroides

Mapa de calor que muestra las variables clave de cada cluster, con los valores originales anotados y el color representando la posición relativa (verde = valor bajo, rojo = valor alto).

> **Diapositiva 9** de la presentación


In [ ]:
# ── Heatmap de centroides: variables clave ────────────────────────────────────
key_feats = ['capacitat_mitja', 'mitja_nab_global', 'std_nab_global',
             'ratio_buida_global', 'ratio_plena_global',
             'ratio_buida_hora_punta', 'ratio_plena_hora_punta',
             'nab_mitja_punta_matinada', 'nab_mitja_punta_tarda',
             'mitja_rang_bicis_norm']

feat_labels = ['Capacitat', 'NAB global', 'Std NAB', 'Ratio buid.',
               'Ratio plena', 'Buid.punta', 'Plena.punta',
               'NAB mat.', 'NAB tard.', 'Rang bicis']

cluster_labels = [f"C{c}: {cluster_names[c]}" for c in range(K_FINAL)]

# Normalizamos cada variable entre 0 y 1 para la visualización
mat = np.zeros((K_FINAL, len(key_feats)))
for j, feat in enumerate(key_feats):
    vals = [centroids_df.loc[c, feat] for c in range(K_FINAL)]
    vmin, vmax = min(vals), max(vals)
    mat[:, j] = [(v - vmin) / (vmax - vmin) if vmax > vmin else 0.5 for v in vals]

# Valores originales para anotar
mat_labels = [[f"{centroids_df.loc[c, feat]:.3f}" for feat in key_feats]
              for c in range(K_FINAL)]

fig, ax = plt.subplots(figsize=(12, 4))
fig.patch.set_facecolor('#F8F9FA'); ax.set_facecolor('#F8F9FA')
im = ax.imshow(mat, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(feat_labels)))
ax.set_xticklabels(feat_labels, rotation=30, ha='right', fontsize=10)
ax.set_yticks(range(K_FINAL))
ax.set_yticklabels(cluster_labels, fontsize=10)

for i in range(K_FINAL):
    for j in range(len(key_feats)):
        color_text = 'white' if mat[i, j] > 0.6 or mat[i, j] < 0.25 else 'black'
        ax.text(j, i, mat_labels[i][j], ha='center', va='center',
                fontsize=8, color=color_text)

plt.colorbar(im, ax=ax, label='Valor normalitzat (0=mínim, 1=màxim)', shrink=.8)
ax.set_title('Centroides per cluster: features clau (valors originals anotats)',
             fontsize=12, fontweight='bold', pad=12, color='#1D3557')

plt.tight_layout()
plt.savefig('png/heatmap_centroids.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardat: png/heatmap_centroids.png')

---
## 10. Distribución de distritos por cluster

Visualizamos cómo se reparten las estaciones de cada cluster entre los distritos de Barcelona, en valores absolutos y porcentuales.


In [ ]:
# ── Tabla cruzada cluster × distrito ─────────────────────────────────────────
crosstab = pd.crosstab(df_valid['cluster_name'], df_valid['nom_districte'])
crosstab_pct = crosstab.div(crosstab.sum(axis=1), axis=0) * 100

print("Estacions per cluster i districte:")
display(crosstab)

In [ ]:
# ── Gráfico de distritos por cluster (absoluto) ──────────────────────────────
short_labels = {
    "Perifèric Baix Ús": "C0-Perifèric",
    "Central Alta Ocupació": "C1-Central",
    "Origen Commuter Matí": "C2-Commuter",
    "Ús Mixt Equilibrat": "C3-Mixt",
    "Residencial Entrada Tarda": "C4-Residencial",
}

crosstab_short = crosstab.copy()
crosstab_short.index = [short_labels.get(i, i) for i in crosstab.index]

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor('#F8F9FA'); ax.set_facecolor('#F8F9FA')
crosstab_short.plot(kind='barh', stacked=True, ax=ax, colormap='tab10')
ax.set_title('Districtes per cluster (absolut)',
             fontsize=13, fontweight='bold', pad=14, color='#1D3557')
ax.set_ylabel('Cluster', fontsize=10)
ax.set_xlabel('Nº estacions', fontsize=10)
ax.tick_params(axis='y', labelsize=10)
ax.legend(title='Districte', bbox_to_anchor=(1.02, 1), loc='upper left',
          borderaxespad=0, fontsize=9, framealpha=.9)
ax.grid(axis='x', ls='--', alpha=.3)

plt.tight_layout()
plt.savefig('png/district_absolut.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardat: png/district_absolut.png')

In [ ]:
# ── Gráfico de distritos por cluster (% relativo) ────────────────────────────
crosstab_pct_short = crosstab_pct.copy()
crosstab_pct_short.index = [short_labels.get(i, i) for i in crosstab_pct.index]

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor('#F8F9FA'); ax.set_facecolor('#F8F9FA')
crosstab_pct_short.plot(kind='barh', stacked=True, ax=ax, colormap='tab10')
ax.set_title('Districtes per cluster (% relatiu)',
             fontsize=13, fontweight='bold', pad=14, color='#1D3557')
ax.set_ylabel('Cluster', fontsize=10)
ax.set_xlabel('%', fontsize=10)
ax.tick_params(axis='y', labelsize=10)
ax.legend(title='Districte', bbox_to_anchor=(1.02, 1), loc='upper left',
          borderaxespad=0, fontsize=9, framealpha=.9)
ax.grid(axis='x', ls='--', alpha=.3)

plt.tight_layout()
plt.savefig('png/district_relatiu.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardat: png/district_relatiu.png')

---
## 11. Mapa interactivo del clustering

Mapa con todas las estaciones clasificadas por cluster. Cada círculo representa una estación y el **color** indica el cluster asignado. Se puede abrir el archivo `html/mapa_clustering.html` directamente en el navegador.

> **Diapositiva 8** de la presentación.


In [ ]:
# ── Mapa interactivo con Folium ──────────────────────────────────────────────
center_lat = df_valid['lat'].mean()
center_lon = df_valid['lon'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=13,
               tiles='CartoDB positron',
               attr='© OpenStreetMap contributors, © CartoDB')

# Añadimos un círculo por cada estación
for c in range(K_FINAL):
    mask = df_valid['cluster'] == c
    sub = df_valid[mask]
    color = COLORS_CLUSTER[c]
    for _, row in sub.iterrows():
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=5,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.8,
            tooltip=(
                f"Estació {int(row['id_estacio'])} | {row['nom_districte']}<br>"
                f"Cluster: C{c} - {cluster_names[c]}<br>"
                f"NAB global: {row['mitja_nab_global']:.3f}<br>"
                f"Ratio buid.: {row['ratio_buida_global']:.3f}<br>"
                f"Ratio plena: {row['ratio_plena_global']:.3f}"
            )
        ).add_to(m)

# Título
m.get_root().html.add_child(folium.Element(f"""
<div style="position:fixed;top:16px;left:50%;transform:translateX(-50%);
    z-index:1000;background:white;padding:9px 22px;border-radius:9px;
    box-shadow:0 2px 8px rgba(0,0,0,.22);font-family:Arial,sans-serif;
    font-size:15px;font-weight:bold;color:#1D3557;pointer-events:none;">
  Bicing Barcelona — Clusters d'estacions (2022–2024)
</div>
"""))

# Leyenda
legend_items = ""
for c in range(K_FINAL):
    n = (df_valid['cluster'] == c).sum()
    legend_items += (
        f'<div style="display:flex;align-items:center;margin-bottom:5px;">'
        f'<div style="width:14px;height:14px;border-radius:50%;'
        f'background:{COLORS_CLUSTER[c]};margin-right:8px;flex-shrink:0;"></div>'
        f'<span style="font-size:12px;">C{c}: {cluster_names[c]} ({n})</span></div>'
    )

legend_html = f"""
<div style="position:fixed; bottom:30px; left:30px; background:white;
     border:1px solid #ccc; border-radius:6px; padding:12px 16px;
     font-family:Arial,sans-serif; box-shadow:2px 2px 6px rgba(0,0,0,0.2);
     z-index:9999;">
    <div style="font-weight:bold;margin-bottom:8px;font-size:13px;">Clusters Bicing</div>
    {legend_items}
</div>
"""
m.get_root().html.add_child(Element(legend_html))

m.save('html/mapa_clusters_interactiu.html')
print('HTML guardat: html/mapa_clusters_interactiu.html')
display(m)

---
## 12. Exportación de los resultados

Guardamos los resultados en archivos CSV y en la base de datos DuckDB.


In [ ]:
# ── Archivo principal: cada estación con su cluster ───────────────────────────
output_cols = ['id_estacio', 'lat', 'lon', 'nom_districte',
               'cluster', 'cluster_name'] + feature_cols
df_valid[output_cols].to_csv('csv/estacio_cluster.csv', index=False)
print(f"Guardat: csv/estacio_cluster.csv ({len(df_valid)} files)")

# ── Guardar en DuckDB ────────────────────────────────────────────────────────
con.register("estacio_df", df_valid)
con.execute("CREATE OR REPLACE TABLE estacio_cluster AS SELECT * FROM estacio_df")
print("Guardat taula estacio_cluster a", DB_PATH)

# ── Centroides ───────────────────────────────────────────────────────────────
centroids_df.to_csv('csv/cluster_centroids.csv', index=False)
print("Guardat: csv/cluster_centroids.csv")

In [ ]:
# ── Tabla resumen por cluster ────────────────────────────────────────────────
summary_rows = []
for c in range(K_FINAL):
    mask = df_valid['cluster'] == c
    sub = df_valid[mask]
    summary_rows.append({
        'cluster': c,
        'cluster_name': cluster_names[c],
        'n_stations': mask.sum(),
        'top_district': sub['nom_districte'].value_counts().index[0],
        'capacitat_mitja_mean': sub['capacitat_mitja'].mean(),
        'nab_global_mean': sub['mitja_nab_global'].mean(),
        'std_nab_mean': sub['std_nab_global'].mean(),
        'ratio_buida_mean': sub['ratio_buida_global'].mean(),
        'ratio_plena_mean': sub['ratio_plena_global'].mean(),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('csv/cluster_summary.csv', index=False)
print("Guardat: csv/cluster_summary.csv")
print()
print("Resum final:")
display(summary_df[['cluster', 'cluster_name', 'n_stations', 'top_district',
                     'nab_global_mean', 'ratio_buida_mean', 'ratio_plena_mean']].round(3))

---
## 13. Cierre

Cerramos la conexión y listamos todos los archivos generados.


In [ ]:
con.close()
print('Connexió a DuckDB tancada.\n')

print('=== Fitxers generats ===')
for carpeta in ['html', 'png', 'csv']:
    fitxers = sorted(glob.glob(f'{carpeta}/*'))
    if fitxers:
        print(f'\n{carpeta.upper()}/  ({len(fitxers)} fitxers)')
        for f in fitxers:
            print(f'  {f}  ({os.path.getsize(f)/1024:.0f} KB)')